# Experiment Bot — Analysis Pipeline
Compute behavioral metrics from bot output and compare to human reference data.

In [ ]:
import hashlib
import json
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
HUMAN_DIR = ROOT / 'data' / 'human'
OUTPUT_DIR = ROOT / 'output'
CACHE_DIR = ROOT / 'cache'

def generate_sub_id(seed_str: str) -> str:
    """Generate a 24-char lowercase hex ID resembling a Prolific subject ID."""
    return hashlib.sha256(seed_str.encode()).hexdigest()[:24]

def load_experiment_data(path):
    """Load experiment data from CSV or JSON."""
    p = Path(path)
    if p.suffix == '.json':
        with open(p) as f:
            return pd.DataFrame(json.load(f))
    return pd.read_csv(p)

def load_cached_config(label):
    """Load cached config and extract Claude's expected parameters."""
    config_path = CACHE_DIR / label / 'config.json'
    if not config_path.exists():
        print(f"  WARNING: No cached config for {label}")
        return None
    with open(config_path) as f:
        return json.load(f)

def display_config_params(config, label):
    """Display Claude's expected behavioral parameters from the cached config."""
    if config is None:
        return
    print(f"=== Claude's Expected Parameters ({label}) ===")
    rd = config.get('response_distributions', {})
    for cond, dist in rd.items():
        p = dist.get('params', {})
        print(f"  {cond}: mu={p.get('mu')}, sigma={p.get('sigma')}, tau={p.get('tau')}")
    perf = config.get('performance', {})
    print(f"  Accuracy targets: {perf.get('accuracy', {})}")
    print(f"  Omission targets: {perf.get('omission_rate', {})}")
    te = config.get('temporal_effects', {})
    enabled = [k for k, v in te.items() if isinstance(v, dict) and v.get('enabled')]
    if enabled:
        print(f"  Temporal effects enabled: {enabled}")
    bsj = config.get('between_subject_jitter', {})
    if bsj.get('rt_mean_sd_ms'):
        print(f"  Between-subject jitter: rt_mean_sd={bsj['rt_mean_sd_ms']}ms")
    print()

def find_latest_run(task_dir_pattern):
    """Find the latest run directory matching a pattern."""
    matches = sorted(OUTPUT_DIR.glob(task_dir_pattern + '/*/experiment_data.*'))
    if not matches:
        return None
    return matches[-1].parent

def find_all_runs(task_dir_pattern):
    """Find ALL run directories matching a pattern, sorted oldest to newest."""
    matches = sorted(OUTPUT_DIR.glob(task_dir_pattern + '/*/experiment_data.*'))
    return [m.parent for m in matches]

def summary_table(df, metrics):
    """Mean +/- SD summary for each metric across subjects."""
    rows = []
    for col in metrics:
        if col not in df.columns:
            rows.append({'metric': col, 'mean': float('nan'), 'sd': float('nan'), 'n': 0})
            continue
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        rows.append({'metric': col, 'mean': round(series.mean(), 4), 'sd': round(series.std(ddof=1), 4), 'n': len(series)})
    return pd.DataFrame(rows).set_index('metric')

print(f"ROOT: {ROOT}")
print(f"Output dirs: {sorted([p.name for p in OUTPUT_DIR.iterdir() if p.is_dir()])}")
for task_dir in sorted(OUTPUT_DIR.iterdir()):
    if task_dir.is_dir():
        runs = [d for d in task_dir.iterdir() if d.is_dir()]
        print(f"  {task_dir.name}: {len(runs)} runs")

## Human Reference Data
Load RDoC behavioral battery data, filter to included sessions.

In [23]:
EXCLUSION_COLS = ['Session-Level Exclusions', 'Task-Level Exclusions', 'Subject-Level Exclusions']

def apply_exclusions(df):
    mask = pd.Series([True] * len(df), index=df.index)
    for col in EXCLUSION_COLS:
        if col in df.columns:
            mask &= df[col].str.strip().eq('Include')
    return df[mask].copy()

ss_human = apply_exclusions(pd.read_csv(HUMAN_DIR / 'stop_signal.csv'))
stroop_human = apply_exclusions(pd.read_csv(HUMAN_DIR / 'stroop.csv'))
stroop_human['stroop_effect'] = (
    pd.to_numeric(stroop_human['incongruent_rt'], errors='coerce')
    - pd.to_numeric(stroop_human['congruent_rt'], errors='coerce')
)

SS_METRICS = ['go_accuracy', 'go_omission_rate', 'go_rt', 'go_rt_all_responses',
              'mean_stop_failure_RT', 'stop_accuracy', 'max_SSD', 'mean_SSD', 'min_SSD', 'final_SSD']
STROOP_METRICS = ['congruent_accuracy', 'congruent_omission_rate', 'congruent_rt',
                  'incongruent_accuracy', 'incongruent_omission_rate', 'incongruent_rt']

print(f"Stop Signal: {len(ss_human)} sessions after exclusions")
print(summary_table(ss_human, SS_METRICS))
print(f"\nStroop: {len(stroop_human)} sessions after exclusions")
print(summary_table(stroop_human, STROOP_METRICS + ['stroop_effect']))

Stop Signal: 2412 sessions after exclusions
                          mean        sd     n
metric                                        
go_accuracy             0.9669    0.0468  2407
go_omission_rate        0.0113    0.0270  2412
go_rt                 648.7588   99.6029  2412
go_rt_all_responses   648.9268   99.8257  2412
mean_stop_failure_RT  571.6641   87.2044  2412
stop_accuracy           0.5214    0.0237  2407
max_SSD               584.5149  133.2118  2412
mean_SSD              414.6373  107.6641  2412
min_SSD               220.1907   88.3784  2412
final_SSD             416.0655  136.8757  2412

Stroop: 2478 sessions after exclusions
                               mean       sd     n
metric                                            
congruent_accuracy           0.9612   0.0465  2477
congruent_omission_rate      0.0042   0.0159  2478
congruent_rt               575.1424  66.8844  2478
incongruent_accuracy         0.9196   0.0632  2477
incongruent_omission_rate    0.0059   0.0172  

## 1. RDoC Stop Signal (ExpFactory)
Load cached config parameters, compute metrics from experiment output.

In [24]:
# Load Claude's config
ss_rdoc_config = load_cached_config('expfactory_stop_signal')
display_config_params(ss_rdoc_config, 'expfactory_stop_signal')

# Find latest run
ss_rdoc_dir = find_latest_run('stop_signal_task_(rdoc)')
if ss_rdoc_dir is None:
    print("No RDoC stop signal data found. Run the bot first.")
else:
    data_file = list(ss_rdoc_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    test_trials = df[df['trial_id'] == 'test_trial']
    assert len(test_trials) == 180, f'Expected 180 test trials, found {len(test_trials)}'

    go_trials = test_trials[test_trials['condition'] == 'go']
    stop_trials = test_trials[test_trials['condition'] == 'stop']
    assert len(go_trials) == 120, f'Expected 120 go trials, found {len(go_trials)}'
    assert len(stop_trials) == 60, f'Expected 60 stop trials, found {len(stop_trials)}'

    correct_go = go_trials[go_trials['correct_trial'] == 1]
    incorrect_stop = stop_trials[stop_trials['correct_trial'] == 0]

    ss_rdoc_metrics = {
        'go_accuracy': len(correct_go) / len(go_trials),
        'go_omission_rate': go_trials['rt'].isna().mean(),
        'go_rt': correct_go['rt'].mean(),
        'go_rt_all_responses': go_trials['rt'].mean(),
        'mean_stop_failure_RT': incorrect_stop['rt'].mean(),
        'stop_accuracy': (stop_trials['correct_trial'] == 1).mean(),
        'max_SSD': stop_trials['SSD'].max(),
        'mean_SSD': stop_trials['SSD'].mean(),
        'min_SSD': stop_trials['SSD'].min(),
        'final_SSD': stop_trials['SSD'].iloc[-1],
    }

    print("Bot metrics:")
    for k, v in ss_rdoc_metrics.items():
        print(f"  {k}: {v:.4f}")

    # Compare to human
    print("\nBot vs Human:")
    for k in SS_METRICS:
        bot_val = ss_rdoc_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(ss_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(ss_human[k], errors='coerce').std()
        within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
        print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")

=== Claude's Expected Parameters (expfactory_stop_signal) ===
  go: mu=430, sigma=60, tau=120
  stop_failure: mu=350, sigma=45, tau=80
  Accuracy targets: {'go': 0.92, 'stop': 0.5}
  Omission targets: {'go': 0.03, 'stop': 0.0}
  Temporal effects enabled: ['autocorrelation', 'fatigue_drift', 'post_error_slowing', 'pink_noise', 'post_interrupt_slowing']
  Between-subject jitter: rt_mean_sd=45ms

Loading: /Users/loganbennett/Downloads/experiment_bot/output/stop_signal_task_(rdoc)/2026-03-22_17-55-26/experiment_data.json
Bot metrics:
  go_accuracy: 0.9083
  go_omission_rate: 0.0917
  go_rt: 622.5688
  go_rt_all_responses: 622.5688
  mean_stop_failure_RT: 566.7333
  stop_accuracy: 0.5000
  max_SSD: 650.0000
  mean_SSD: 403.3333
  min_SSD: 250.0000
  final_SSD: 300.0000

Bot vs Human:
  go_accuracy                     bot=   0.908  human=   0.967 ±  0.047  within_1SD=NO
  go_omission_rate                bot=   0.092  human=   0.011 ±  0.027  within_1SD=NO
  go_rt                           bo

## 2. RDoC Stroop (ExpFactory)
Load cached config parameters, compute metrics from experiment output.

In [25]:
stroop_rdoc_config = load_cached_config('expfactory_stroop')
display_config_params(stroop_rdoc_config, 'expfactory_stroop')

stroop_rdoc_dir = find_latest_run('stroop_color-word_task_(rdoc)')
if stroop_rdoc_dir is None:
    print("No RDoC Stroop data found. Run the bot first.")
else:
    data_file = list(stroop_rdoc_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    test_trials = df[df['trial_id'] == 'test_trial']
    congruent = test_trials[test_trials['condition'] == 'congruent']
    incongruent = test_trials[test_trials['condition'] == 'incongruent']
    assert len(test_trials) == 120, f'Expected 120 test trials, found {len(test_trials)}'
    assert len(congruent) == 60, f'Expected 60 congruent, found {len(congruent)}'
    assert len(incongruent) == 60, f'Expected 60 incongruent, found {len(incongruent)}'

    stroop_rdoc_metrics = {
        'congruent_accuracy': (congruent['correct_trial'] == 1).mean(),
        'congruent_omission_rate': congruent['rt'].isna().mean(),
        'congruent_rt': congruent[congruent['correct_trial'] == 1]['rt'].mean(),
        'incongruent_accuracy': (incongruent['correct_trial'] == 1).mean(),
        'incongruent_omission_rate': incongruent['rt'].isna().mean(),
        'incongruent_rt': incongruent[incongruent['correct_trial'] == 1]['rt'].mean(),
    }

    print("Bot metrics:")
    for k, v in stroop_rdoc_metrics.items():
        print(f"  {k}: {v:.4f}")

    stroop_effect = stroop_rdoc_metrics['incongruent_rt'] - stroop_rdoc_metrics['congruent_rt']
    print(f"  stroop_effect: {stroop_effect:.4f}")

    print("\nBot vs Human:")
    for k in STROOP_METRICS:
        bot_val = stroop_rdoc_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(stroop_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(stroop_human[k], errors='coerce').std()
        within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
        print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")
    h_se_mean = stroop_human['stroop_effect'].mean()
    h_se_sd = stroop_human['stroop_effect'].std()
    within = "YES" if abs(stroop_effect - h_se_mean) < h_se_sd else "NO"
    print(f"  {'stroop_effect':30s}  bot={stroop_effect:8.3f}  human={h_se_mean:8.3f} \u00b1{h_se_sd:7.3f}  within_1SD={within}")

=== Claude's Expected Parameters (expfactory_stroop) ===
  congruent: mu=520, sigma=60, tau=120
  incongruent: mu=590, sigma=70, tau=160
  Accuracy targets: {'congruent': 0.97, 'incongruent': 0.9}
  Omission targets: {'congruent': 0.01, 'incongruent': 0.02}
  Temporal effects enabled: ['autocorrelation', 'fatigue_drift', 'post_error_slowing', 'condition_repetition', 'pink_noise']
  Between-subject jitter: rt_mean_sd=45ms

Loading: /Users/loganbennett/Downloads/experiment_bot/output/stroop_color-word_task_(rdoc)/2026-03-22_18-04-37/experiment_data.csv
Bot metrics:
  congruent_accuracy: 0.9833
  congruent_omission_rate: 0.0000
  congruent_rt: 651.8475
  incongruent_accuracy: 0.7500
  incongruent_omission_rate: 0.0500
  incongruent_rt: 742.6667
  stroop_effect: 90.8192

Bot vs Human:
  congruent_accuracy              bot=   0.983  human=   0.961 ±  0.046  within_1SD=YES
  congruent_omission_rate         bot=   0.000  human=   0.004 ±  0.016  within_1SD=YES
  congruent_rt                  

## 3. STOP-IT Stop Signal
Load cached config parameters, compute metrics from experiment output.

In [26]:
stopit_config = load_cached_config('stopit_stop_signal')
display_config_params(stopit_config, 'stopit_stop_signal')

stopit_dir = find_latest_run('stop*signal*task*(stop-it)')
if stopit_dir is None:
    print("No STOP-IT data found. Run the bot first.")
else:
    data_file = list(stopit_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    go_trials = df[df['signal'] == 'no']
    stop_trials = df[df['signal'] == 'yes']
    assert len(go_trials) == 216, f'Expected 216 go trials, found {len(go_trials)}'
    assert len(stop_trials) == 72, f'Expected 72 stop trials, found {len(stop_trials)}'

    stopit_metrics = {
        'go_accuracy': go_trials['correct'].mean(),
        'go_omission_rate': (go_trials['response'] == 'undefined').mean(),
        'go_rt': go_trials[go_trials['correct'] == True]['rt'].mean(),
        'go_rt_all_responses': go_trials['rt'].mean(),
        'mean_stop_failure_RT': stop_trials[stop_trials['correct'] == False]['rt'].mean(),
        'stop_accuracy': stop_trials['correct'].mean(),
        'max_SSD': stop_trials['SSD'].max(),
        'mean_SSD': stop_trials['SSD'].mean(),
        'min_SSD': stop_trials['SSD'].min(),
        'final_SSD': stop_trials.iloc[-1]['SSD'],
    }

    print("Bot metrics:")
    for k, v in stopit_metrics.items():
        print(f"  {k}: {v:.4f}")

    print("\nBot vs Human (ExpFactory reference):")
    for k in SS_METRICS:
        bot_val = stopit_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(ss_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(ss_human[k], errors='coerce').std()
        within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
        print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")

=== Claude's Expected Parameters (stopit_stop_signal) ===
  go_left: mu=420, sigma=50, tau=90
  go_right: mu=420, sigma=50, tau=90
  failed_stop: mu=340, sigma=40, tau=60
  Accuracy targets: {'go_left': 0.96, 'go_right': 0.96, 'stop_signal': 0.5}
  Omission targets: {'go_left': 0.01, 'go_right': 0.01, 'stop_signal': 0.0}
  Temporal effects enabled: ['autocorrelation', 'fatigue_drift', 'post_error_slowing', 'condition_repetition', 'pink_noise', 'post_interrupt_slowing']
  Between-subject jitter: rt_mean_sd=45ms

Loading: /Users/loganbennett/Downloads/experiment_bot/output/stop_signal_task_(stop-it)/2026-03-22_18-10-57/experiment_data.csv
Bot metrics:
  go_accuracy: 0.8750
  go_omission_rate: 0.0880
  go_rt: 4383.3915
  go_rt_all_responses: 4239.2081
  mean_stop_failure_RT: 552.7647
  stop_accuracy: 0.5278
  max_SSD: 650.0000
  mean_SSD: 394.4444
  min_SSD: 100.0000
  final_SSD: 450.0000

Bot vs Human (ExpFactory reference):
  go_accuracy                     bot=   0.875  human=   0.967 

## 4. Cognition.run Stroop
Load cached config parameters, compute metrics from experiment output.

In [27]:
cogrun_config = load_cached_config('cognitionrun_stroop')
display_config_params(cogrun_config, 'cognitionrun_stroop')

cogrun_dir = find_latest_run('stroop_color-word_task')
# Exclude rdoc variant
if cogrun_dir and '(rdoc)' in str(cogrun_dir):
    dirs = sorted(OUTPUT_DIR.glob('stroop_color-word_task/*/experiment_data.*'))
    cogrun_dir = dirs[-1].parent if dirs else None

if cogrun_dir is None:
    print("No Cognition.run Stroop data found. Run the bot first.")
else:
    data_file = list(cogrun_dir.glob('experiment_data.*'))[0]
    print(f"Loading: {data_file}")
    df = load_experiment_data(data_file)

    test_trials = df[df['text'].notna()].copy()
    assert len(test_trials) == 15, f'Expected 15 test trials, found {len(test_trials)}'

    test_trials['trial_condition'] = test_trials.apply(
        lambda row: 'congruent' if str(row['text']).lower() == str(row['colour']).lower() else 'incongruent', axis=1
    )
    test_trials['correct_trial'] = test_trials.apply(
        lambda row: 1 if str(row['response']).lower() == str(row['colour'])[0].lower() else 0, axis=1
    )

    congruent = test_trials[test_trials['trial_condition'] == 'congruent']
    incongruent = test_trials[test_trials['trial_condition'] == 'incongruent']

    cogrun_metrics = {
        'congruent_accuracy': congruent['correct_trial'].mean() if len(congruent) > 0 else float('nan'),
        'congruent_omission_rate': congruent['rt'].isna().mean() if len(congruent) > 0 else float('nan'),
        'congruent_rt': congruent[congruent['correct_trial'] == 1]['rt'].mean() if len(congruent) > 0 else float('nan'),
        'incongruent_accuracy': incongruent['correct_trial'].mean() if len(incongruent) > 0 else float('nan'),
        'incongruent_omission_rate': incongruent['rt'].isna().mean() if len(incongruent) > 0 else float('nan'),
        'incongruent_rt': incongruent[incongruent['correct_trial'] == 1]['rt'].mean() if len(incongruent) > 0 else float('nan'),
    }

    print(f"Bot metrics ({len(congruent)} congruent, {len(incongruent)} incongruent):")
    for k, v in cogrun_metrics.items():
        print(f"  {k}: {v:.4f}" if not np.isnan(v) else f"  {k}: N/A")

    if not np.isnan(cogrun_metrics['congruent_rt']) and not np.isnan(cogrun_metrics['incongruent_rt']):
        stroop_effect = cogrun_metrics['incongruent_rt'] - cogrun_metrics['congruent_rt']
        print(f"  stroop_effect: {stroop_effect:.4f}")

    print("\nBot vs Human (ExpFactory reference):")
    for k in STROOP_METRICS:
        bot_val = cogrun_metrics.get(k, float('nan'))
        h_mean = pd.to_numeric(stroop_human[k], errors='coerce').mean()
        h_sd = pd.to_numeric(stroop_human[k], errors='coerce').std()
        if np.isnan(bot_val):
            print(f"  {k:30s}  bot=     N/A  human={h_mean:8.3f} \u00b1{h_sd:7.3f}")
        else:
            within = "YES" if abs(bot_val - h_mean) < h_sd else "NO"
            print(f"  {k:30s}  bot={bot_val:8.3f}  human={h_mean:8.3f} \u00b1{h_sd:7.3f}  within_1SD={within}")

=== Claude's Expected Parameters (cognitionrun_stroop) ===
  congruent: mu=580, sigma=80, tau=150
  incongruent: mu=680, sigma=95, tau=200
  Accuracy targets: {'congruent': 0.97, 'incongruent': 0.92}
  Omission targets: {'congruent': 0.005, 'incongruent': 0.01}
  Temporal effects enabled: ['autocorrelation', 'fatigue_drift', 'post_error_slowing', 'condition_repetition', 'pink_noise']
  Between-subject jitter: rt_mean_sd=60ms

Loading: /Users/loganbennett/Downloads/experiment_bot/output/stroop_color-word_task/2026-03-22_19-05-59/experiment_data.csv
Bot metrics (7 congruent, 8 incongruent):
  congruent_accuracy: 1.0000
  congruent_omission_rate: 0.0000
  congruent_rt: 852.0000
  incongruent_accuracy: 1.0000
  incongruent_omission_rate: 0.0000
  incongruent_rt: 931.6250
  stroop_effect: 79.6250

Bot vs Human (ExpFactory reference):
  congruent_accuracy              bot=   1.000  human=   0.961 ±  0.046  within_1SD=YES
  congruent_omission_rate         bot=   0.000  human=   0.004 ±  0.016

## Export Bot Metrics as CSV
Process **all** runs (not just the latest) for each task. Save per-run metrics in the same format as the human reference data.

In [ ]:
OUTPUT_CSV_DIR = ROOT / 'data' / 'bot'
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

def compute_ss_rdoc_metrics(run_dir):
    """Compute stop signal metrics for one RDoC ExpFactory run."""
    data_file = list(run_dir.glob('experiment_data.*'))[0]
    df = load_experiment_data(data_file)
    test_trials = df[df['trial_id'] == 'test_trial']
    if len(test_trials) != 180:
        print(f"  WARNING: {run_dir.name} has {len(test_trials)} test trials (expected 180), skipping")
        return None
    go = test_trials[test_trials['condition'] == 'go']
    stop = test_trials[test_trials['condition'] == 'stop']
    correct_go = go[go['correct_trial'] == 1]
    incorrect_stop = stop[stop['correct_trial'] == 0]
    return {
        'go_accuracy': len(correct_go) / len(go),
        'go_omission_rate': go['rt'].isna().mean(),
        'go_rt': correct_go['rt'].mean(),
        'go_rt_all_responses': go['rt'].mean(),
        'mean_stop_failure_RT': incorrect_stop['rt'].mean(),
        'stop_accuracy': (stop['correct_trial'] == 1).mean(),
        'max_SSD': stop['SSD'].max(),
        'mean_SSD': stop['SSD'].mean(),
        'min_SSD': stop['SSD'].min(),
        'final_SSD': stop['SSD'].iloc[-1],
    }

def compute_stopit_metrics(run_dir):
    """Compute stop signal metrics for one STOP-IT run."""
    data_file = list(run_dir.glob('experiment_data.*'))[0]
    df = load_experiment_data(data_file)
    go = df[df['signal'] == 'no']
    stop = df[df['signal'] == 'yes']
    if len(go) != 216 or len(stop) != 72:
        print(f"  WARNING: {run_dir.name} has {len(go)} go / {len(stop)} stop (expected 216/72), skipping")
        return None
    return {
        'go_accuracy': go['correct'].mean(),
        'go_omission_rate': (go['response'] == 'undefined').mean(),
        'go_rt': go[go['correct'] == True]['rt'].mean(),
        'go_rt_all_responses': go['rt'].mean(),
        'mean_stop_failure_RT': stop[stop['correct'] == False]['rt'].mean(),
        'stop_accuracy': stop['correct'].mean(),
        'max_SSD': stop['SSD'].max(),
        'mean_SSD': stop['SSD'].mean(),
        'min_SSD': stop['SSD'].min(),
        'final_SSD': stop.iloc[-1]['SSD'],
    }

def compute_stroop_rdoc_metrics(run_dir):
    """Compute Stroop metrics for one RDoC ExpFactory run."""
    data_file = list(run_dir.glob('experiment_data.*'))[0]
    df = load_experiment_data(data_file)
    test_trials = df[df['trial_id'] == 'test_trial']
    cong = test_trials[test_trials['condition'] == 'congruent']
    incong = test_trials[test_trials['condition'] == 'incongruent']
    if len(test_trials) != 120:
        print(f"  WARNING: {run_dir.name} has {len(test_trials)} test trials (expected 120), skipping")
        return None
    return {
        'congruent_accuracy': (cong['correct_trial'] == 1).mean(),
        'congruent_omission_rate': cong['rt'].isna().mean(),
        'congruent_rt': cong[cong['correct_trial'] == 1]['rt'].mean(),
        'incongruent_accuracy': (incong['correct_trial'] == 1).mean(),
        'incongruent_omission_rate': incong['rt'].isna().mean(),
        'incongruent_rt': incong[incong['correct_trial'] == 1]['rt'].mean(),
    }

def compute_cogrun_stroop_metrics(run_dir):
    """Compute Stroop metrics for one Cognition.run run."""
    data_file = list(run_dir.glob('experiment_data.*'))[0]
    df = load_experiment_data(data_file)
    test_trials = df[df['text'].notna()].copy()
    if len(test_trials) != 15:
        print(f"  WARNING: {run_dir.name} has {len(test_trials)} test trials (expected 15), skipping")
        return None
    test_trials['trial_condition'] = test_trials.apply(
        lambda row: 'congruent' if str(row['text']).lower() == str(row['colour']).lower() else 'incongruent', axis=1
    )
    test_trials['correct_trial'] = test_trials.apply(
        lambda row: 1 if str(row['response']).lower() == str(row['colour'])[0].lower() else 0, axis=1
    )
    cong = test_trials[test_trials['trial_condition'] == 'congruent']
    incong = test_trials[test_trials['trial_condition'] == 'incongruent']
    return {
        'congruent_accuracy': cong['correct_trial'].mean() if len(cong) > 0 else float('nan'),
        'congruent_omission_rate': cong['rt'].isna().mean() if len(cong) > 0 else float('nan'),
        'congruent_rt': cong[cong['correct_trial'] == 1]['rt'].mean() if len(cong) > 0 else float('nan'),
        'incongruent_accuracy': incong['correct_trial'].mean() if len(incong) > 0 else float('nan'),
        'incongruent_omission_rate': incong['rt'].isna().mean() if len(incong) > 0 else float('nan'),
        'incongruent_rt': incong[incong['correct_trial'] == 1]['rt'].mean() if len(incong) > 0 else float('nan'),
    }

# --- Process ALL runs ---

ss_rows = []
stroop_rows = []

# RDoC Stop Signal
for run_dir in find_all_runs('stop_signal_task_(rdoc)'):
    metrics = compute_ss_rdoc_metrics(run_dir)
    if metrics:
        ss_rows.append({
            'sub_id': generate_sub_id(f"ss_rdoc_{run_dir.name}"),
            'date_time': run_dir.name.replace('_', ' ', 1).replace('-', '/', 2),
            'session': 1, 'platform': 'expfactory', **metrics,
        })
print(f"RDoC Stop Signal: {len([r for r in ss_rows if r['platform'] == 'expfactory'])} runs")

# STOP-IT
for run_dir in find_all_runs('stop*signal*task*(stop-it)'):
    metrics = compute_stopit_metrics(run_dir)
    if metrics:
        ss_rows.append({
            'sub_id': generate_sub_id(f"stopit_{run_dir.name}"),
            'date_time': run_dir.name.replace('_', ' ', 1).replace('-', '/', 2),
            'session': 1, 'platform': 'stopit', **metrics,
        })
print(f"STOP-IT: {len([r for r in ss_rows if r['platform'] == 'stopit'])} runs")

# RDoC Stroop
for run_dir in find_all_runs('stroop_color-word_task_(rdoc)'):
    metrics = compute_stroop_rdoc_metrics(run_dir)
    if metrics:
        stroop_rows.append({
            'sub_id': generate_sub_id(f"stroop_rdoc_{run_dir.name}"),
            'date_time': run_dir.name.replace('_', ' ', 1).replace('-', '/', 2),
            'session': 1, 'platform': 'expfactory', **metrics,
        })
print(f"RDoC Stroop: {len([r for r in stroop_rows if r['platform'] == 'expfactory'])} runs")

# Cognition.run Stroop
for run_dir in find_all_runs('stroop_color-word_task'):
    if '(rdoc)' in str(run_dir):
        continue
    metrics = compute_cogrun_stroop_metrics(run_dir)
    if metrics:
        stroop_rows.append({
            'sub_id': generate_sub_id(f"cogrun_{run_dir.name}"),
            'date_time': run_dir.name.replace('_', ' ', 1).replace('-', '/', 2),
            'session': 1, 'platform': 'cognitionrun', **metrics,
        })
print(f"Cognition.run Stroop: {len([r for r in stroop_rows if r['platform'] == 'cognitionrun'])} runs")

# --- Save CSVs ---

if ss_rows:
    ss_bot_df = pd.DataFrame(ss_rows)
    col_order = ['sub_id', 'date_time', 'session', 'platform'] + SS_METRICS
    ss_bot_df = ss_bot_df[[c for c in col_order if c in ss_bot_df.columns]]
    ss_bot_df.to_csv(OUTPUT_CSV_DIR / 'stop_signal.csv', index=False)
    print(f"\nSaved {len(ss_bot_df)} stop signal rows to data/bot/stop_signal.csv")
    print(ss_bot_df.to_string(index=False))
else:
    print("\nNo stop signal bot data to export.")

if stroop_rows:
    stroop_bot_df = pd.DataFrame(stroop_rows)
    col_order = ['sub_id', 'date_time', 'session', 'platform'] + STROOP_METRICS
    stroop_bot_df = stroop_bot_df[[c for c in col_order if c in stroop_bot_df.columns]]
    stroop_bot_df.to_csv(OUTPUT_CSV_DIR / 'stroop.csv', index=False)
    print(f"\nSaved {len(stroop_bot_df)} Stroop rows to data/bot/stroop.csv")
    print(stroop_bot_df.to_string(index=False))
else:
    print("\nNo Stroop bot data to export.")

## Summary
All metrics computed and exported. Bot CSVs saved to `data/bot/` in the same format as human reference data.